# Named Entity Recognition (NER) Model Training

Running the cells below will evaluate and train an NER model using the annotated MIMIC-IV data. 

## Imports

In [1]:
import csv
import json
import logging
import os
import random
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import spacy
import medspacy
from medspacy.sentence_splitting import PyRuSHSentencizer

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import KFold, train_test_split
from scipy.stats import iqr 

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

from transformers import (
    AdamW,
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    BertForTokenClassification,
    BertTokenizerFast,
    get_linear_schedule_with_warmup,
    RobertaConfig,
    RobertaForTokenClassification,
    RobertaTokenizer,
    RobertaTokenizerFast,
)

import wandb


# Load and convert dataset

Starting with the annotations from label studio, we first convert them to CONL2003 format. 

In [ ]:
def tokenize_with_offsets(text):
    """
    Tokenizes text while preserving character offsets.
    Returns a list of tuples: (token, start_offset, end_offset).
    """
    tokens = []
    for match in re.finditer(r'\w+|[^\w\s]|\n', text):
        token = match.group(0)
        tokens.append((token, match.start(), match.end()))
    return tokens

def assign_labels_with_offsets(tokens, annotations):
    """
    Assigns BIO labels to tokens based on character offsets from annotations.
    """
    labels = ['O'] * len(tokens)
    for annotation in annotations:
        start, end = annotation['start'], annotation['end']
        label_type = annotation['labels'][0]
        for i, (token, token_start, token_end) in enumerate(tokens):
            if token == '\n':  # Skip newline tokens
                continue
            if token_start >= start and token_end <= end:
                labels[i] = f'B-{label_type}' if token_start == start else f'I-{label_type}'
            elif (token_start < start < token_end) or (token_start < end < token_end):
                labels[i] = f'I-{label_type}'
    return labels

def apply_conll_formatting(json_data):
    """
    Converts JSON Label Studio annotations to CoNLL-2003 format.
    Each document begins with a marker line containing the document ID.
    """
    conll_data = []
    for entry in json_data:
        doc_id = entry['id']
        
        # Ensure the text is not empty before processing
        assert entry['text'], f"""Error: Document ID {doc_id} has empty text. Please re-associate annotations 
                with MIMIC-IV note texts before training."""
                
        document = [f"# New Document ID: {doc_id}"]
        text = entry['text']
        annotations = entry['label']
        tokens_with_offsets = tokenize_with_offsets(text)
        tokens = [token for token, _, _ in tokens_with_offsets]
        labels = assign_labels_with_offsets(tokens_with_offsets, annotations)
        
        for token, label in zip(tokens, labels):
            if token == '\n':  # Use blank lines for sentence breaks
                document.append("")
            else:
                document.append(f"{token} {label}")
        document.append("")  # Ensure a blank line at the end of the document
        conll_data.append(document)
    return conll_data

def save_to_file(data, file_path):
    """
    Saves the formatted CoNLL data to a file.
    """
    with open(file_path, 'w') as f:
        for document in data:
            for line in document:
                f.write(line + '\n')

# Load JSON data from the Label Studio clinical NER annotations
file_path = 'data/ner/entity_annotations.json'
with open(file_path, 'r') as file:
    json_data = json.load(file)

# Convert JSON data to CoNLL-2003 format with document markers
formatted_data = apply_conll_formatting(json_data)

# Save the full formatted data
data_file_path = 'data/ner/entity_annotations_conll.txt'
save_to_file(formatted_data, data_file_path)
print(f"Full data saved to {data_file_path}")


# Generate training chunks

The documents are too large for the RoBERTa model, hence they are converted into chunks of max 512 tokens.

In [ ]:
# === MODEL & TOKENIZER CONFIGURATION ===
MODEL_DIR = "data/models/RoBERTa-base-PM-M3-Voc-distill-align-hf"
VOCAB_FILE = f"{MODEL_DIR}/vocab.json"
MERGES_FILE = f"{MODEL_DIR}/merges.txt"

tokenizer = RobertaTokenizerFast(
    vocab_file=VOCAB_FILE, 
    merges_file=MERGES_FILE, 
    add_prefix_space=True
)

MAX_SEQ_LEN = 512
MAX_TOKEN_LIMIT = MAX_SEQ_LEN - 2  # Reserve space for special tokens

def load_ner_data(file_path):
    """Load a NER dataset from a CoNLL-formatted file with document markers.
    Returns a list of (document, document_id) tuples.
    """
    documents = []
    current_document = []
    current_sentence = []
    current_doc_id = None
    total_annotations = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('# New Document ID:'):
                if current_document and current_doc_id is not None:
                    if current_sentence:
                        current_document.append(current_sentence)
                        current_sentence = []
                    documents.append((current_document, current_doc_id))
                    current_document = []
                parts = line.split(':', maxsplit=1)
                current_doc_id = parts[1].strip() if len(parts) > 1 else "UNKNOWN_ID"
                continue
            elif line == "":
                if current_sentence:
                    current_document.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split(' ')
                token = parts[0]
                label = ' '.join(parts[1:])
                if label.startswith('B-'):
                    total_annotations += 1
                current_sentence.append((token, label))
    if current_sentence:
        current_document.append(current_sentence)
    if current_document and current_doc_id is not None:
        documents.append((current_document, current_doc_id))
    print(f"Total Number of Annotations: {total_annotations}")
    return documents

def merge_spanning_entities(sentences):
    """Merge sentences if an entity spans across sentence boundaries.
    Returns the updated list of sentences.
    """
    merged_sentences = []
    i = 0
    while i < len(sentences):
        if i < len(sentences) - 1:
            last_label = sentences[i][-1][1]
            next_first_label = sentences[i + 1][0][1]
            if (last_label.startswith(('B-', 'I-')) and 
                next_first_label.startswith('I-') and 
                last_label.split('-', 1)[1] == next_first_label.split('-', 1)[1]):
                merged_sentence = sentences[i] + sentences[i + 1]
                merged_sentences.append(merged_sentence)
                i += 2
                continue
        merged_sentences.append(sentences[i])
        i += 1
    return merged_sentences

def count_subword_tokens(sentence):
    """Count subword tokens in a sentence using the RoBERTa tokenizer.
    Returns the token count as an integer.
    """
    tokens = [word for word, _ in sentence]
    encoding = tokenizer(tokens, is_split_into_words=True, add_special_tokens=False)
    return len(encoding.input_ids)

def find_split_index(sentence):
    """Find a suitable index to split a sentence that exceeds token limits.
    Returns an index (int) at which to split.
    """
    length = len(sentence)
    mid = length // 2
    radius = min(mid, length - mid - 1)
    for offset in range(radius + 1):
        for index in [mid + offset, mid - offset]:
            if 0 <= index < length:
                token, label = sentence[index]
                if token in {',', '.'} and label == 'O':
                    return index
    for offset in range(radius + 1):
        for index in [mid + offset, mid - offset]:
            if 0 < index < length - 1:
                _, current_label = sentence[index]
                _, next_label = sentence[index + 1]
                if not next_label.startswith('I-'):
                    return index
    return mid

def split_sentence(sentence, max_token_limit=MAX_TOKEN_LIMIT):
    """Recursively split a sentence into segments under the token limit.
    Returns a list of sentence segments.
    """
    if count_subword_tokens(sentence) <= max_token_limit:
        return [sentence]
    split_idx = find_split_index(sentence)
    left_segment = sentence[:split_idx + 1]
    right_segment = sentence[split_idx + 1:]
    return split_sentence(left_segment, max_token_limit) + split_sentence(right_segment, max_token_limit)

def split_long_sentences(document, max_token_limit=MAX_TOKEN_LIMIT):
    """Split sentences in a document that exceed the token limit.
    Returns the document with all sentences within the token limit.
    """
    split_document = []
    for sentence in document:
        split_document.extend(split_sentence(sentence, max_token_limit))
    return split_document

def chunk_document(document, max_token_limit=MAX_TOKEN_LIMIT):
    """Chunk sentences into groups that fit within the token limit.
    Returns a list of chunks (each a list of (token, label) pairs).
    """
    chunks = []
    current_chunk = []
    for sentence in document:
        tentative_chunk = current_chunk + sentence
        if count_subword_tokens(tentative_chunk) > max_token_limit:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence
        else:
            current_chunk = tentative_chunk
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

def extract_entities(document):
    """Extract contiguous entities from a document based on BIO labels.
    Returns a list of entities (each entity is a list of tokens).
    """
    entities = []
    current_entity = []
    current_entity_type = None
    for sentence in document:
        for token, label in sentence:
            if label.startswith('B-'):
                if current_entity:
                    entities.append(current_entity)
                current_entity = [token]
                current_entity_type = label.split('-', 1)[1]
            elif label.startswith('I-'):
                label_type = label.split('-', 1)[1]
                if current_entity and current_entity_type == label_type:
                    current_entity.append(token)
                else:
                    if current_entity:
                        entities.append(current_entity)
                    current_entity = [token]
                    current_entity_type = label_type
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = []
                    current_entity_type = None
    if current_entity:
        entities.append(current_entity)
    return entities


"""
Main processing: load, merge, split, and chunk the NER data.
Computes and prints various dataset statistics.
"""
input_file = 'data/ner/entity_annotations_conll.txt'
documents_with_ids = load_ner_data(input_file)
processed_documents = []
entities_per_document = []
entity_word_lengths = []
for document, doc_id in documents_with_ids:
    merged_document = merge_spanning_entities(document)
    entities = extract_entities(merged_document)
    entities_per_document.append(len(entities))
    entity_word_lengths.extend(len(entity) for entity in entities)
    split_document = split_long_sentences(merged_document, max_token_limit=MAX_TOKEN_LIMIT)
    chunks = chunk_document(split_document, max_token_limit=MAX_TOKEN_LIMIT)
    processed_documents.append((chunks, doc_id))

# Combine all chunks into a single list
train_chunks = []
for chunks, _ in processed_documents:
    train_chunks.extend(chunks)

print(f"Number of training chunks: {len(train_chunks)}")

Total Number of Annotations: 85328
Number of training chunks: 2525


# Dataset statistics

In [7]:

median_entities = np.median(entities_per_document)
iqr_entities = iqr(entities_per_document)
median_entity_length = np.median(entity_word_lengths)
iqr_entity_length = iqr(entity_word_lengths)
print(f"Median number of entities per document: {median_entities} ± IQR: {iqr_entities}")
print(f"Median entity length (in words): {median_entity_length} ± IQR: {iqr_entity_length}")
q1_entities = np.percentile(entities_per_document, 25)
q3_entities = np.percentile(entities_per_document, 75)
q1_entity_length = np.percentile(entity_word_lengths, 25)
q3_entity_length = np.percentile(entity_word_lengths, 75)
print(f"Entities per document => Q1: {q1_entities}, Q3: {q3_entities}")
print(f"Entity length (in words) => Q1: {q1_entity_length}, Q3: {q3_entity_length}")

Median number of entities per document: 206.5 ± IQR: 136.25
Median entity length (in words): 2.0 ± IQR: 2.0
Entities per document => Q1: 137.0, Q3: 273.25
Entity length (in words) => Q1: 1.0, Q3: 3.0


# Training/Eval Setup

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(21)

print(torch.version.cuda)  # This gives you the CUDA version PyTorch was built with
print(torch.cuda.is_available())  # This returns True if a GPU with CUDA support is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Chosen RoBERTa-PM model (e.g. "RoBERTa-large-PM-M3-Voc-hf")
CHOSEN_MODEL = "data/models/RoBERTa-base-PM-M3-Voc-distill-align-hf" 

12.1
True
cuda


In [9]:
# Dataset class for loading and transforming data to get a format suitable for the model
class NERDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_len):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        
        label_to_id = {"O": 0, 
                "B-procedure": 1, 
                "I-procedure": 2,
                "B-health_context": 3, 
                "I-health_context": 4,                
                "B-disorder": 5, 
                "I-disorder": 6,
                "B-normal_finding": 7, 
                "I-normal_finding": 8,
                "B-abnormal_finding": 9, 
                "I-abnormal_finding": 10,
                "B-medication": 11, 
                "I-medication": 12
                }

        sentence = self.sentences[idx]
        tokens = [token for token, label in sentence]
        labels = [label_to_id[label] for token, label in sentence]

        # Tokenize sentence
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        word_ids = encoding.word_ids()
        encoded_labels = [-100] * self.max_len

        previous_word_idx = None
        for i, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != previous_word_idx:
                # This is the first token of the word
                encoded_labels[i] = labels[word_idx]
            else:
                # This is a subtoken, mark it as "inside" the entity
                original_label = sentence[word_idx][1]
                if original_label.startswith('B-'):
                    encoded_labels[i] = label_to_id['I-' + original_label[2:]]
                else:
                    encoded_labels[i] = label_to_id[original_label]

            previous_word_idx = word_idx

        item = {key: torch.squeeze(val) for key, val in encoding.items()}
        item['word_ids'] = torch.tensor([w if w is not None else -1 for w in word_ids], dtype=torch.long)  # Replace None with -1
        item['labels'] = torch.tensor(encoded_labels, dtype=torch.long)

        return item

## Training Definition

In [10]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Paths to vocabulary and merges files
vocab_file = os.path.join(CHOSEN_MODEL, "vocab.json")
merges_file = os.path.join(CHOSEN_MODEL, "merges.txt")

tokenizer = RobertaTokenizerFast(
    vocab_file=vocab_file,
    merges_file=merges_file,
    add_prefix_space=True
)

config = RobertaConfig.from_pretrained(CHOSEN_MODEL)
config.num_labels = 13
config.architectures = ["RobertaForTokenClassification"]

def create_optimizer_and_scheduler(model, train_loader, epochs, learning_rate, weight_decay, warmup_ratio=0.1, gradient_accumulation_steps=1):
    """Create optimizer and scheduler for training.
    Returns an AdamW optimizer and a linear warmup scheduler.
    """
    total_steps = (len(train_loader) // gradient_accumulation_steps) * epochs
    warmup_steps = int(warmup_ratio * total_steps)
    
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    return optimizer, scheduler

def train_epoch(model, dataloader, optimizer, scheduler, device, epoch, max_grad_norm=1.0, log_interval=96, gradient_accumulation_steps=4):
    """Train model for one epoch with gradient accumulation and clipping.
    Returns the average training loss for the epoch.
    """
    model.to(device)
    model.train()
    total_loss = 0.0
    start_time = time.time()
    
    logging.info(f"Epoch {epoch + 1} training started.")
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / gradient_accumulation_steps
        total_loss += loss.item() * gradient_accumulation_steps
        
        loss.backward()
        
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        if batch_idx % log_interval == 0:
            elapsed = time.time() - start_time
            current_lr = scheduler.get_last_lr()[0]
            logging.info(
                f"Batch {batch_idx}/{len(dataloader)} - Loss: {loss.item() * gradient_accumulation_steps:.4f} - "
                f"LR: {current_lr:.6e} - Time Elapsed: {elapsed:.2f}s"
            )
    
    avg_loss = total_loss / len(dataloader)
    logging.info(f"Epoch {epoch + 1} training completed - Average Loss: {avg_loss:.4f}")
    return avg_loss

## Evaluation Definition

In [11]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from collections import defaultdict
import torch

id_to_label = {
    0: "O",
    1: "B-procedure",
    2: "I-procedure",
    3: "B-health_context",
    4: "I-health_context",
    5: "B-disorder",
    6: "I-disorder",
    7: "B-normal_finding",
    8: "I-normal_finding",
    9: "B-abnormal_finding",
    10: "I-abnormal_finding",
    11: "B-medication",
    12: "I-medication"
}

def merge_subtokens(word_ids, labels):
    """Merge subtoken labels to word-level labels.
    Skips special tokens (-1) and repeated subtokens.
    """
    merged_labels = []
    seen_word_ids = set()
    for i, word_idx in enumerate(word_ids):
        if word_idx == -1 or word_idx in seen_word_ids:
            continue
        merged_labels.append(labels[i])
        seen_word_ids.add(word_idx)
    return merged_labels


def get_entities(labels, batch_num, chunk_num):
    """Extract entities from a BIO label sequence and tag them.
    Associates each entity with its batch and chunk indices.
    """
    entities = []
    entity = None
    for i, label in enumerate(labels):
        if label.startswith("B-"):
            if entity:
                entities.append(entity)
            entity = [label[2:], i, i, batch_num, chunk_num]
        elif label.startswith("I-"):
            if entity and label[2:] == entity[0]:
                entity[2] = i
            else:
                if entity:
                    entities.append(entity)
                entity = [label[2:], i, i, batch_num, chunk_num]
        else:
            if entity:
                entities.append(entity)
                entity = None
    if entity:
        entities.append(entity)
    return entities


def evaluate(model, dataloader, device, id_to_label):
    """Evaluate model performance and compute metrics.
    Returns loss, average F1, per-entity precision, recall, F1, confusion matrix, and report.
    """
    model.eval()
    total_loss = 0
    true_entities, pred_entities = [], []
    all_true_labels, all_pred_labels = [], []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss, logits = outputs.loss, outputs.logits
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=-1)
            
            true_labels = labels.cpu().numpy()
            pred_labels = predictions.cpu().numpy()
            word_ids = batch['word_ids'].cpu().numpy()
            
            for i in range(len(true_labels)):
                valid_indices = true_labels[i] != -100
                true_labels_filtered = true_labels[i][valid_indices]
                pred_labels_filtered = pred_labels[i][valid_indices]
                word_ids_filtered = word_ids[i][valid_indices]
                true_labels_str = [id_to_label[label] for label in true_labels_filtered]
                pred_labels_str = [id_to_label[label] for label in pred_labels_filtered]
                merged_true_labels = merge_subtokens(word_ids_filtered, true_labels_str)
                merged_pred_labels = merge_subtokens(word_ids_filtered, pred_labels_str)
                all_true_labels.extend(merged_true_labels)
                all_pred_labels.extend(merged_pred_labels)
                chunk_num = i
                true_entities.extend(get_entities(merged_true_labels, batch_idx, chunk_num))
                pred_entities.extend(get_entities(merged_pred_labels, batch_idx, chunk_num))
    
    avg_loss = total_loss / len(dataloader)
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    
    true_entity_set = {(label, start, end, batch, chunk) for label, start, end, batch, chunk in true_entities}
    pred_entity_set = {(label, start, end, batch, chunk) for label, start, end, batch, chunk in pred_entities}
    
    for entity in pred_entity_set:
        if entity in true_entity_set:
            tp[entity[0]] += 1
        else:
            fp[entity[0]] += 1
    for entity in true_entity_set:
        if entity not in pred_entity_set:
            fn[entity[0]] += 1

    precision, recall, f1 = {}, {}, {}
    for entity_type in (tp.keys() | fp.keys() | fn.keys()):
        if entity_type == 'O':
            continue
        precision[entity_type] = (
            tp[entity_type] / (tp[entity_type] + fp[entity_type])
            if (tp[entity_type] + fp[entity_type]) > 0 else 0.0
        )
        recall[entity_type] = (
            tp[entity_type] / (tp[entity_type] + fn[entity_type])
            if (tp[entity_type] + fn[entity_type]) > 0 else 0.0
        )
        if precision[entity_type] + recall[entity_type] > 0:
            f1[entity_type] = 2 * (precision[entity_type] * recall[entity_type]) / (
                precision[entity_type] + recall[entity_type]
            )
        else:
            f1[entity_type] = 0.0

    avg_f1 = np.mean(list(f1.values()))
    all_labels = list(id_to_label.values())
    confusion_mat = confusion_matrix(all_true_labels, all_pred_labels, labels=all_labels)
    non_o_labels = [label for label in id_to_label.values() if label != "O"]
    report = classification_report(all_true_labels, all_pred_labels, labels=non_o_labels, digits=4)
    
    return avg_loss, avg_f1, precision, recall, f1, confusion_mat, report


## Early stopping definition

In [12]:
class EarlyStopping:
    def __init__(
        self, 
        enabled=True,         
        patience=5, 
        verbose=False, 
        delta=0, 
        path='checkpoint.pt', 
        trace_func=print
    ):
        """
        Args:
            enabled (bool): If False, early stopping won't be used.
            patience (int): How long to wait after last time the validation F1 improved.
            verbose (bool): If True, prints a message for each F1 improvement.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
            path (str): Path for the checkpoint to be saved.
            trace_func (function): trace print function.
        """
        self.enabled = enabled    
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.f1_score_max = float('-inf')
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, f1_score, model):
        # If early stopping is disabled, do nothing and return
        if not self.enabled:
            return

        if self.best_score is None:
            self.best_score = f1_score
            self.save_checkpoint(f1_score, model)
        elif f1_score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'Early Stopping Counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = f1_score
            self.save_checkpoint(f1_score, model)
            self.counter = 0

    def save_checkpoint(self, f1_score, model):
        '''Saves model when validation F1 score improves.'''
        if self.verbose:
            self.trace_func(
                f'Validation F1 score increased ({self.f1_score_max:.6f} --> {f1_score:.6f}).  Saving model ...'
            )
        torch.save(model.state_dict(), self.path)
        self.f1_score_max = f1_score


## Model Hyperparameters

In [ ]:
batch_size = 8
max_batch_size = 8 # For gradient-accumulation if your GPU can't handle the full batch size
max_len = 512 
learning_rate = 3e-5 
weight_decay = 0.01
max_grad_norm = 1.0
epochs = 12
warmup_ratio = 0.1 

use_early_stopping = True

if batch_size % max_batch_size != 0:
    raise ValueError(f"batch_size ({batch_size}) must be divisible by max_batch_size ({max_batch_size})")

gradient_accumulation_steps = batch_size // max_batch_size 

# Directory to save the models
save_dir = "../data/models/ner_model"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

fold_metrics = []
num_labels = 13  

# Cross-validation

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import os
import torch
import logging
from collections import defaultdict

num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=21)

fold_metrics = []
overall_precision = defaultdict(list)
overall_recall = defaultdict(list)
overall_f1 = defaultdict(list)

wandb.init(
    project="mimic_ner",
)

for fold_idx, (train_indices, val_indices) in enumerate(kf.split(train_chunks)):
    logging.info(f"Starting fold {fold_idx + 1}/{num_folds}")
    
    train_chunks_fold = [train_chunks[i] for i in train_indices]
    val_chunks_fold = [train_chunks[i] for i in val_indices]
    
    train_dataset = NERDataset(train_chunks_fold, tokenizer, max_len)
    val_dataset = NERDataset(val_chunks_fold, tokenizer, max_len)
    
    train_loader = DataLoader(train_dataset, batch_size=max_batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=max_batch_size, shuffle=False)
    
    checkpoint_name = f'checkpoint_fold{fold_idx+1}.pt'
    early_stopping = EarlyStopping(
        enabled=use_early_stopping,
        patience=5,
        verbose=True,
        path=os.path.join(save_dir, checkpoint_name)
    )
    
    model = RobertaForTokenClassification.from_pretrained(CHOSEN_MODEL, config=config)
    optimizer, scheduler = create_optimizer_and_scheduler(
        model, train_loader, epochs, learning_rate, weight_decay, warmup_ratio, gradient_accumulation_steps
    )
    
    for epoch in range(epochs):
        logging.info(f"Epoch {epoch + 1}/{epochs} for fold {fold_idx + 1}")
        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            epoch,
            max_grad_norm=max_grad_norm,
            gradient_accumulation_steps=gradient_accumulation_steps
        )
        
        val_loss, val_f1, precision, recall, f1, confusion_mat, report = evaluate(
            model, val_loader, device, id_to_label
        )
        logging.info(f"Validation F1 Score (Strict, Macro): {val_f1:.4f}")
        
        wandb.log({
            "fold": fold_idx + 1,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_f1": val_f1
        })
        
        early_stopping(val_f1, model)
        if early_stopping.early_stop:
            logging.info("Early stopping triggered")
            break
    
    if use_early_stopping:
        model.load_state_dict(torch.load(os.path.join(save_dir, checkpoint_name)))
    else:
        model.load_state_dict(torch.load(os.path.join(save_dir, f"final_model_no_early_stop.pt")))
    
    val_loss, val_f1, precision, recall, f1, confusion_mat, report = evaluate(
        model, val_loader, device, id_to_label
    )
    fold_metrics.append({
        'fold': fold_idx + 1,
        'val_loss': val_loss,
        'val_f1': val_f1,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })
    
    # Accumulate metrics across folds for each entity type
    for entity_type in f1.keys():
        overall_precision[entity_type].append(precision[entity_type])
        overall_recall[entity_type].append(recall[entity_type])
        overall_f1[entity_type].append(f1[entity_type])


In [ ]:
# After training all folds, calculate the average and standard deviation for metrics across folds
avg_val_loss = np.mean([m['val_loss'] for m in fold_metrics])
std_val_loss = np.std([m['val_loss'] for m in fold_metrics])

avg_val_f1 = np.mean([m['val_f1'] for m in fold_metrics])
std_val_f1 = np.std([m['val_f1'] for m in fold_metrics])

labels = [
    "O",
    "B-procedure",
    "I-procedure",
    "B-health_context",
    "I-health_context",
    "B-disorder",
    "I-disorder",
    "B-normal_finding",
    "I-normal_finding",
    "B-abnormal_finding",
    "I-abnormal_finding",
    "B-medication",
    "I-medication"
]

cm_df = pd.DataFrame(confusion_mat, index=labels, columns=labels)

logging.info("=======Overall Results=======")
logging.info(f"\n{cm_df}")
logging.info(f"\n{report}")
logging.info(
    f"Average Validation Loss over {num_folds} folds: {avg_val_loss:.6f} ± {std_val_loss:.6f}"
)
logging.info(
    f"Average Validation F1 over {num_folds} folds: {avg_val_f1:.4f} ± {std_val_f1:.4f}"
)

logging.info("\nAverage Entity-wise precision, recall, and F1 across all folds:")
for entity_type in sorted(overall_f1.keys()):
    avg_precision = np.mean(overall_precision[entity_type])
    std_precision = np.std(overall_precision[entity_type])

    avg_recall = np.mean(overall_recall[entity_type])
    std_recall = np.std(overall_recall[entity_type])

    avg_f1 = np.mean(overall_f1[entity_type])
    std_f1 = np.std(overall_f1[entity_type])

    logging.info(
        f"{entity_type}: Precision: {avg_precision:.4f} ± {std_precision:.4f}, "
        f"Recall: {avg_recall:.4f} ± {std_recall:.4f}, "
        f"F1: {avg_f1:.4f} ± {std_f1:.4f}"
    )
    # ---- Log to W&B ----
    wandb.log({
        f"avg_precision_{entity_type}": avg_precision,
        f"std_precision_{entity_type}": std_precision,
        f"avg_recall_{entity_type}": avg_recall,
        f"std_recall_{entity_type}": std_recall,
        f"avg_f1_{entity_type}": avg_f1,
        f"std_f1_{entity_type}": std_f1,
    })

2025-02-15 19:05:50,504 - INFO - =======Overall Results=======
2025-02-15 19:05:50,508 - INFO - 
                         O  B-procedure  I-procedure  B-health_context  \
O                   129441          102          252                91   
B-procedure             81         1365           39                25   
I-procedure            175           52         2704                 4   
B-health_context        84           32            2              1272   
I-health_context       258            5           82                77   
B-disorder              40            3            1                46   
I-disorder             123            0            1                 7   
B-normal_finding       103            3            0                 6   
I-normal_finding       239            2           11                 0   
B-abnormal_finding     134            8            4                28   
I-abnormal_finding     609            6           68                 1   
B-medication   

# Re-train on all data and save model

In [ ]:
# Retrain on the full dataset (train_chunks) using the chosen hyperparameters

all_dataset = NERDataset(train_chunks, tokenizer, max_len)
all_loader = DataLoader(all_dataset, batch_size=max_batch_size, shuffle=True)

model = RobertaForTokenClassification.from_pretrained(CHOSEN_MODEL, config=config)
optimizer, scheduler = create_optimizer_and_scheduler(
    model, all_loader, epochs, learning_rate, weight_decay, warmup_ratio, gradient_accumulation_steps
)

for epoch in range(epochs):
    logging.info(f"Epoch {epoch + 1}/{epochs} (re-training on all data)")
    train_loss = train_epoch(
        model,
        all_loader,
        optimizer,
        scheduler,
        device,
        epoch,
        max_grad_norm=max_grad_norm,
        gradient_accumulation_steps=gradient_accumulation_steps
    )
    logging.info(f"Training loss: {train_loss:.4f}")
    wandb.log({
        "epoch_all_data": epoch + 1,
        "train_loss_all_data": train_loss,
    })

# Save the final model for inference
inference_model_path = os.path.join(save_dir, "final_model_for_inference.pt")
torch.save(model.state_dict(), inference_model_path)
model.config.to_json_file(os.path.join(save_dir, "model_config.json"))
tokenizer.save_pretrained(save_dir)


Some weights of the model checkpoint at ../RoBERTa-base-PM-M3-Voc-distill-align-hf were not used when initializing RobertaForTokenClassification: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.decoder.weight', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at ../RoBERTa-base-PM-M3-Voc-distill-al

('Roberta_NER_MIMIC_CV\\tokenizer_config.json',
 'Roberta_NER_MIMIC_CV\\special_tokens_map.json',
 'Roberta_NER_MIMIC_CV\\vocab.json',
 'Roberta_NER_MIMIC_CV\\merges.txt',
 'Roberta_NER_MIMIC_CV\\added_tokens.json',
 'Roberta_NER_MIMIC_CV\\tokenizer.json')